<a href="https://colab.research.google.com/github/prajwaljshetty/side-quests/blob/main/agents/campus-pilot/CampusPilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CampusPilot

Smart College Campus Assistant

CampusPilot uses CrewAI to classify user requests and route them to the appropriate response. For event-planning requests, specialised agents work in sequence to create a schedule, review logistics, and prepare one final response.

## Assignment objectives

- Classify user intent before taking action.
- Handle greetings and unclear requests without running the event-planning workflow.
- Ask for required information when it is missing.
- Coordinate specialised agents for schedules, logistics, and announcements.
- Consolidate the agents' work into one clear response.

## Workflow

1. Receive the user's request.
2. Classify intent and extract explicit details.
3. Respond directly to greetings and out-of-scope requests.
4. Clarify unclear requests.
5. Run the event-planning agents for event requests.
6. Display the final plan.


## 1. Install dependencies

Install the libraries required by this notebook. Run this cell first.

In [2]:
%pip install -q crewai openai requests pydantic

## 2. Import libraries
All Python imports are grouped here so dependencies are easy to review.

In [3]:
from google.colab import userdata
from openai import OpenAI
from crewai import LLM, Agent, Task, Crew, Process
from crewai.tools import tool
from pydantic import BaseModel, Field
from typing import Literal, Optional
from datetime import datetime, timedelta
import json
import re
import requests

## 3. Configure the language model

Read the API key from Colab Secrets, verify the endpoint, and initialise the model used by all agents.

In [5]:
API_KEY = userdata.get("nexus_api_key")
BASE_URL = "https://nexusapi.navigatelabs.ai"
MODEL_NAME = "llama3-8b-instruct"

if not API_KEY:
    raise ValueError(
        "Missing Colab Secret 'nexus_api_key'. Add it in Colab Secrets and enable notebook access."
    )

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

available_models = [item.id for item in client.models.list().data]
print("Nexus connection successful.")
print("Available models:", ", ".join(available_models))

if MODEL_NAME not in available_models:
    print(f"Note: {MODEL_NAME!r} was not listed. Update MODEL_NAME to an available model ID.")

llm = LLM(
    model=MODEL_NAME,
    base_url=BASE_URL,
    api_key=API_KEY,
    custom_openai=True,
    temperature=0.2
)

Nexus connection successful.
Available models: nova-micro, mistral-7b-instruct, llama3-8b-instruct, gpt-4.1-nano, text-embedding-3-small, whisper-1, gemini-2.5-flash, gpt-4o-mini-tts, nano-banana


## Tool: Schedule Calculator

Purpose: calculate activity end times from a start time and duration, keeping timetable arithmetic consistent.

In [6]:
@tool("Schedule Calculator")
def schedule_calculator(start_time: str, duration_minutes: int) -> str:
    """Calculate an activity end time. start_time must use 24-hour HH:MM format."""
    if duration_minutes <= 0:
        return "Invalid duration: use a positive number of minutes."
    try:
        start = datetime.strptime(start_time, "%H:%M")
    except ValueError:
        return "Invalid start time: use 24-hour HH:MM format, for example 09:30."
    end = start + timedelta(minutes=duration_minutes)
    next_day = " (next day)" if end.date() > start.date() else ""
    return f"Start: {start:%H:%M}; End: {end:%H:%M}{next_day}; Duration: {duration_minutes} minutes."

## Tool: Resource Estimator

Purpose: estimate basic quantities such as chairs, water bottles, and registration desks from participant count.

In [7]:
@tool("Resource Estimator")
def resource_estimator(participants: int) -> str:
    """Estimate chairs, water bottles (two per participant), and registration desks (one per 100 participants, rounded up)."""
    if participants <= 0:
        return "Invalid participant count: it must be positive."
    return (
        f"Planning estimates for {participants} participants: "
        f"{participants} chairs, {participants * 2} water bottles, "
        f"{(participants + 99) // 100} registration desk(s). "
        "These are estimates; confirm actual requirements with campus staff."
    )

## Tool: Campus Weather

Purpose: retrieve current weather observations near Moodbidri. This is not a forecast for a future event date.

In [8]:
@tool("Campus Weather")
def campus_weather() -> str:
    """Fetch current weather observations near Moodbidri, Karnataka; this is not a future event forecast."""
    try:
        response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": 13.1833,
                "longitude": 74.9333,
                "current": "temperature_2m,precipitation",
                "timezone": "Asia/Kolkata"
            },
            timeout=10
        )
        response.raise_for_status()
        data = response.json()
        current = data.get("current", {})
        units = data.get("current_units", {})
        return (
            f"Observation time: {current.get('time', 'unavailable')}; "
            f"temperature: {current.get('temperature_2m', 'unavailable')} "
            f"{units.get('temperature_2m', '°C')}; precipitation: "
            f"{current.get('precipitation', 'unavailable')} "
            f"{units.get('precipitation', 'mm')}. "
            "This is current weather, not an event-date forecast."
        )
    except (requests.RequestException, ValueError) as exc:
        return f"Weather unavailable: {exc}"

## Intent decision schema

Purpose: define and validate the structured fields returned by the classifier.

In [9]:
class IntentDecision(BaseModel):
    intent: Literal["GREETING", "EVENT_PLANNING", "CLARIFICATION", "OUT_OF_SCOPE"]
    confidence: float = Field(ge=0.0, le=1.0)
    reason: str
    response: str
    participants: Optional[int] = None
    event_type: Optional[str] = None
    missing_details: list[str] = Field(default_factory=list)

## Agent: Intent Classifier

Purpose: identify the user's intent, extract explicitly supplied details, and choose the next action. It must not plan the event.

In [10]:
intent_classifier = Agent(
    role="CampusPilot Intent Classifier",
    goal="Understand the user's message and choose the safest useful next action.",
    backstory=(
        "You are the front door of CampusPilot. You distinguish conversation from "
        "campus event planning, detect ambiguity, extract explicit details, and never "
        "assume every message is an event request. You must not plan the event yourself."
    ),
    llm=llm,
    verbose=False
)

## Task: Classify the user request

Purpose: return one JSON decision without creating a timetable or logistics plan.

In [11]:
intent_task = Task(
    description="""
Classify this user message: {user_request}

Return one JSON object with these exact fields:
intent, confidence, reason, response, participants, event_type, missing_details.

Choose one intent:
- GREETING: greetings, thanks, or ordinary social conversation.
- EVENT_PLANNING: the user clearly asks to organize or plan a campus/college event.
- CLARIFICATION: the intended action is unclear, or the message is too vague to act on safely.
- OUT_OF_SCOPE: the request is unrelated to campus event planning.

Decision rules:
- "hi how are u" is GREETING.
- "Plan a technical fest for 300 students" is EVENT_PLANNING; participants=300.
- "Help me organise something" is CLARIFICATION.
- A clear event request can be EVENT_PLANNING even when details such as date or venue are missing.
- Extract participant count only if the user explicitly mentions it. Otherwise use null.
- List essential missing details only when they block the next useful step.
- Keep response concise, friendly, and professional.
- Do not generate a timetable or logistics plan.
- Do not wrap the JSON in Markdown fences.
""",
    expected_output="One JSON object matching the specified intent decision fields.",
    agent=intent_classifier
)

## Crew: Intent Classification

Purpose: run the classifier task. The orchestrator uses its result to decide whether specialist agents should run.

In [12]:
intent_crew = Crew(
    agents=[intent_classifier],
    tasks=[intent_task],
    process=Process.sequential,
    verbose=False
)

## Agent: Event Planner

Purpose: create a practical timetable and resource estimate. Uses the schedule and resource tools and labels assumptions.

In [13]:
event_planner = Agent(
    role="College Event Planner",
    goal="Create a practical, time-consistent event schedule and resource estimate.",
    backstory=(
        "You plan campus programmes, sessions, competitions, breaks, and closing remarks. "
        "Use tools for calculations. Label assumptions and never invent confirmed facts."
    ),
    tools=[schedule_calculator, resource_estimator],
    llm=llm,
    verbose=False
)

## Agent: Campus Coordinator

Purpose: review venue, equipment, staffing, catering, accessibility, and weather considerations.

In [14]:
campus_coordinator = Agent(
    role="Campus Logistics Coordinator",
    goal="Prepare a realistic logistics checklist and identify arrangements requiring confirmation.",
    backstory=(
        "You consider venue capacity, accessibility, guest reception, equipment, staffing, "
        "catering, and contingencies. Recommendations are not confirmed bookings."
    ),
    tools=[campus_weather],
    llm=llm,
    verbose=False
)

## Agent: Communication Manager

Purpose: consolidate the schedule and logistics into one concise user-facing response.

In [15]:
communication_manager = Agent(
    role="Campus Communication Manager",
    goal="Deliver one concise, polished, consistent campus event response.",
    backstory=(
        "You write clear, welcoming announcements. Use only supplied or established details; "
        "mark unknown date, venue, guest, and registration information as To be confirmed."
    ),
    llm=llm,
    verbose=False
)

## Task: Build the event schedule

Purpose: create the timetable, estimates, assumptions, and pending details.

In [16]:
schedule_task = Task(
    description="""
Plan an event from the user's request: {user_request}
Expected participants: {participants}

Create a one-day timetable with named activities, start/end times, breaks, and closing.
Use Schedule Calculator for time arithmetic and Resource Estimator for planning estimates.
State assumptions and missing information. Do not invent confirmed bookings.
""",
    expected_output="A timetable, resource estimates, assumptions, and details to confirm.",
    agent=event_planner
)

## Task: Review campus logistics

Purpose: identify operational requirements and arrangements that still need confirmation.

In [17]:
logistics_task = Task(
    description="""
Prepare logistics for this event: {user_request}
Expected participants: {participants}

Use the timetable from the previous task. Use Campus Weather and report the observation
time if available. Explicitly say that current weather is not an event-date forecast.
List venue, equipment, catering, staffing, accessibility, and guest arrangements to confirm.
Do not claim any arrangement is booked.
""",
    expected_output="A logistics checklist, weather caveat, and pending confirmations.",
    agent=campus_coordinator,
    context=[schedule_task]
)

## Task: Produce the final response

Purpose: combine schedule and logistics into one polished response.

In [18]:
communication_task = Task(
    description="""
Create the final user-facing answer for this request: {user_request}
Expected participants: {participants}

Combine the schedule and logistics outputs into a concise, readable response:
1. Event overview
2. Timetable
3. Logistics and resource estimates
4. Short announcement
5. Details still to confirm

Keep times and participant counts consistent. Do not invent dates, venues, guest names,
bookings, or registration links. Mark unknown details "To be confirmed".
Use a polished, minimal, Apple-like communication style: clear headings, concise sentences,
useful spacing, no filler, no internal agent commentary.
""",
    expected_output="One clear, polished consolidated campus event plan.",
    agent=communication_manager,
    context=[schedule_task, logistics_task]
)

## Crew: Event Planning

Purpose: execute the event tasks sequentially: Event Planner, Campus Coordinator, then Communication Manager.

In [19]:
campus_crew = Crew(
    agents=[event_planner, campus_coordinator, communication_manager],
    tasks=[schedule_task, logistics_task, communication_task],
    process=Process.sequential,
    verbose=False
)

print("Intent router and event-planning Crew are ready.")

Intent router and event-planning Crew are ready.


## Utilities: Parse and validate

Purpose: parse classifier JSON, validate its schema, and validate event inputs before agent execution.

In [20]:
def parse_intent_result(raw_text: str) -> IntentDecision:
    text = raw_text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    try:
        payload = json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError("Intent classifier did not return JSON.")
        payload = json.loads(match.group())

    decision = IntentDecision.model_validate(payload)
    return decision


def validate_event_input(user_request: str, participants: int) -> Optional[str]:
    if not isinstance(user_request, str) or not user_request.strip():
        return "Please describe the event you want to plan."
    if not isinstance(participants, int) or isinstance(participants, bool):
        return "Participant count must be a whole number."
    if participants <= 0:
        return "Participant count must be greater than zero."
    return None

## Function: Classify a request

Purpose: run the intent crew asynchronously to avoid synchronous CrewAI calls inside Colab's event loop.

In [21]:
async def classify_request(user_request: str) -> IntentDecision:
    result = await intent_crew.kickoff_async(
        inputs={"user_request": user_request}
    )
    raw_text = getattr(result, "raw", str(result))
    return parse_intent_result(raw_text)

## Orchestrator: CampusPilot

Purpose: route greetings, unclear requests, and out-of-scope messages separately. Only event-planning requests proceed to the specialist crew.

In [22]:
async def run_campuspilot():
    user_request = input("You: ").strip()

    if not user_request:
        print("CampusPilot: Tell me what you need help with.")
        return

    print("\nCampusPilot is thinking...")

    greeting_pattern = (
        r"(hi+|hey+|hello+|hy+|yo+|sup|what'?s up|whats up|"
        r"how are you|how r u|how are u|how's it going|thanks|thank you)"
        r"[!.?\s]*"
    )
    if re.fullmatch(greeting_pattern, user_request.lower()):
        print("CampusPilot: Hey! I'm doing well, thanks for asking. What can I help you with?")
        return

    try:
        decision = await classify_request(user_request)
    except Exception:
        import traceback
        print("CampusPilot: I couldn't interpret that request right now. Please try again.")
        print("\nDiagnostic traceback:")
        traceback.print_exc()
        return

    # Conservative routing: low-confidence classifications are clarified, not executed.
    if decision.confidence < 0.60:
        print("CampusPilot:", decision.response or "Could you clarify what you'd like help with?")
        return

    if decision.intent == "GREETING":
        print("CampusPilot:", decision.response)
        return

    if decision.intent == "CLARIFICATION":
        print("CampusPilot:", decision.response or "Could you clarify what you'd like help with?")
        return

    if decision.intent == "OUT_OF_SCOPE":
        print(
            "CampusPilot:",
            decision.response or
            "I can help with campus event schedules, logistics, resources, and announcements."
        )
        return

    if decision.intent != "EVENT_PLANNING":
        print("CampusPilot: I couldn't determine the right next step. Could you rephrase that?")
        return

    if decision.participants is None:
        while True:
            raw_count = input("Expected participants: ").strip()
            try:
                participants = int(raw_count)
            except ValueError:
                print("Enter a whole number, such as 150.")
                continue
            if participants <= 0:
                print("Participant count must be greater than zero.")
                continue
            break
    else:
        participants = decision.participants

    error = validate_event_input(user_request, participants)
    if error:
        print("CampusPilot:", error)
        return

    print("\nPreparing your campus plan...")

    try:
        result = await campus_crew.kickoff_async(
            inputs={
                "user_request": user_request,
                "participants": participants
            }
        )
        print("\n" + "=" * 64)
        print("CAMPUSPILOT")
        print("Your campus event plan")
        print("=" * 64)
        print(result)
    except Exception:
        import traceback
        print("CampusPilot: I couldn't complete the plan right now. Please try again.")
        print("\nDiagnostic traceback:")
        traceback.print_exc()

## Run CampusPilot

Run this cell once per message. It uses await because the orchestration is asynchronous.

In [25]:
await run_campuspilot()

You: plan a event on this weekend of 200 students



CampusPilot is thinking...



Preparing your campus plan...



CAMPUSPILOT
Your campus event plan


**Weekend Campus Fiesta**

**Event Overview**

Join us for a fun-filled day of activities, breaks, and closing remarks at the Weekend Campus Fiesta! This one-day event is designed to bring together 200 students for a mix of icebreaker games, interactive workshops, and student competitions.

**Timetable**

Here is the proposed timetable for the Weekend Campus Fiesta:

**9:00 AM - 9:30 AM:** Registration and Welcome (30 minutes)

* Activity: Student registration and welcome address
* Resource Estimate: 1 staff member, 1 table, 1 chair, and 1 microphone

**9:30 AM - 10:30 AM:** Icebreaker Session (1 hour)

* Activity: Icebreaker games and activities to get students mingling
* Resource Estimate: 2 staff members, 1 sound system, and 1 projector

**10:30 AM - 11:00 AM:** Break (30 minutes)

* Activity: Refreshments and socializing
* Resource Estimate: 1 staff member, 1 table, and 1 coffee machine

**11:00 AM - 12:30 PM:** Workshop Session (1.5 hours)

* 

## Manual test cases

Run the input cell once per test.

- `hy` should return a greeting without calling the LLM.
- `hi how are you` should return a greeting.
- `Plan a technical fest for 300 students` should enter the event-planning workflow.
- `Help me organise something` should ask for clarification.
- `Write a C program for binary search` should be identified as outside the assistant's scope.
- Empty input should be rejected locally.

## Notes and limitations

- The classifier is an LLM and may make mistakes. Its output is schema-validated.
- The confidence score is the model's self-assessment, not a calibrated probability.
- Current weather observations are not future forecasts.
- Resource estimates are illustrative and should be confirmed with campus staff.
- The event crew runs sequentially: Event Planner, Campus Coordinator, Communication Manager.